# Notebook 11: Distributed Data Parallel

## Why This Matters

Distributed Data Parallel (DDP) is the standard method for scaling training across multiple GPUs or nodes. When you move from a single GPU to 8 GPUs, you should see roughly 7-8x throughput improvement -- but only if DDP is set up correctly. Common mistakes -- not using `DistributedSampler`, synchronising incorrectly across ranks, or leaving model state on the wrong device -- cause training to be either incorrect (all workers train on the same data), slower than a single GPU (synchronisation overhead not hidden by compute), or subtly wrong (BatchNorm statistics not properly synchronised).

## Background

DDP uses the **data parallelism** paradigm: the model is replicated on every device, and each replica processes a different shard of the data. After the backward pass, gradients are synchronised across all replicas via **all-reduce**: every device computes its local gradients, then the sum (or mean) is broadcast so all replicas update their parameters identically.

The key efficiency insight: gradient all-reduce is **overlapped with the backward pass**. As soon as a layer's gradient is computed, DDP starts communicating it, while the rest of the backward continues. This hides most of the communication latency.

DDP vs Model Parallelism:
- **DDP**: full model on each device; shard the data. Works when the model fits on one GPU.
- **Model Parallelism / Pipeline Parallelism**: shard the model across devices. Needed when the model is too large for one GPU (use with FSDP or DeepSpeed for large models).

This notebook shows the patterns using a single-process simulation, then explains the multi-GPU launch mechanism.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset, DistributedSampler
import numpy as np
import matplotlib.pyplot as plt
import os

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

# Check distributed backend availability
print(f"NCCL available: {dist.is_nccl_available()}")
print(f"Gloo available: {dist.is_gloo_available()}")
print("Ready.")

## 1. What DDP Does -- Conceptual Model

Understanding the DDP lifecycle prevents all the most common bugs.

**Initialisation** (`init_process_group`):
- Establishes a communication backend (NCCL for GPU, Gloo for CPU)
- Assigns each process a unique `rank` (0 to world_size-1) and a `local_rank` (rank within a node)

**Model wrapping** (`DistributedDataParallel(model)`):
- Broadcasts initial parameters from rank 0 to all other ranks (ensures identical initialisation)
- Registers backward hooks that fire all-reduce on each gradient bucket as it becomes ready

**Forward + backward**:
- Each rank runs forward on its local shard of data
- Backward: as gradients accumulate bucket-by-bucket, DDP fires all-reduce in the background
- After `loss.backward()` completes, all ranks have the same gradient (the global average)

**Optimizer step**:
- Because all ranks have identical gradients, all optimizer updates are identical
- Parameters remain synchronised without any extra communication

In [ ]:
# Simulate DDP's communication pattern conceptually (no actual distributed process needed)
# We show what gradient synchronisation achieves

torch.manual_seed(42)

# Simulate 2 ranks, each with a local gradient
def simulate_allreduce(grad_rank0, grad_rank1, reduction="mean"):
    if reduction == "mean":
        synced = (grad_rank0 + grad_rank1) / 2
    else:
        synced = grad_rank0 + grad_rank1
    return synced, synced  # both ranks get the same value

# Each rank processes a different batch
model = nn.Linear(4, 2)
X_rank0 = torch.randn(8, 4)
X_rank1 = torch.randn(8, 4)
y = torch.randint(0, 2, (8,))

# Forward + backward on each rank independently
def get_local_grad(model, X, y):
    m = nn.Linear(4, 2)
    m.weight.data.copy_(model.weight.data)
    m.bias.data.copy_(model.bias.data)
    loss = F.cross_entropy(m(X), y)
    loss.backward()
    return m.weight.grad.clone(), m.bias.grad.clone()

g_w0, g_b0 = get_local_grad(model, X_rank0, y)
g_w1, g_b1 = get_local_grad(model, X_rank1, y)

# What DDP does: all-reduce
synced_w, _ = simulate_allreduce(g_w0, g_w1)
synced_b, _ = simulate_allreduce(g_b0, g_b1)

# What you'd get if you ran the full batch at once
X_full = torch.cat([X_rank0, X_rank1])
y_full = torch.cat([y, y])
g_w_full, g_b_full = get_local_grad(model, X_full, y_full)

print("DDP all-reduce result vs full-batch gradient:")
print(f"  weight grad match: {torch.allclose(synced_w, g_w_full, atol=1e-5)}")
print(f"  bias   grad match: {torch.allclose(synced_b, g_b_full, atol=1e-5)}")
print("DDP gradient = mean of per-rank gradients = gradient on full virtual batch")

## 2. init_process_group and DDP Wrapping

In a real multi-GPU setup, each process calls `init_process_group` to establish the communication group. The `backend` parameter selects the communication library:

- **nccl**: NVIDIA Collective Communications Library. Best for GPU-to-GPU via NVLink or InfiniBand. Required for production GPU training.
- **gloo**: CPU-based fallback. Works on any hardware including CPU-only nodes. Used for CPU training or debugging.

The `local_rank` environment variable (set by `torchrun`) tells each process which GPU to use. Always move the model to `local_rank` BEFORE wrapping with DDP.

In [ ]:
# DDP setup pattern (runs in single-process simulation mode here)
# In production, this runs inside torchrun-launched processes

def setup_ddp_simulation():
    # In real DDP this would be:
    # local_rank = int(os.environ["LOCAL_RANK"])
    # dist.init_process_group(backend="nccl")
    # torch.cuda.set_device(local_rank)
    
    # For this notebook: single process, CPU simulation
    if not dist.is_initialized():
        os.environ["MASTER_ADDR"] = "localhost"
        os.environ["MASTER_PORT"] = "29500"
        dist.init_process_group(
            backend="gloo",
            rank=0,
            world_size=1,
        )
    return 0  # local_rank

local_rank = setup_ddp_simulation()
print(f"Process group initialised: rank={dist.get_rank()}, world_size={dist.get_world_size()}")

# Model setup: always move to device BEFORE DDP wrapping
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 8)
        self.fc3 = nn.Linear(8, 4)

    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

model_ddp_base = SimpleModel().to(device)
# DDP wrap: broadcasts rank-0 params to all ranks (world_size=1 here, so no-op)
model_ddp = DDP(model_ddp_base)

print(f"DDP model type: {type(model_ddp)}")
print(f"Access underlying module: {type(model_ddp.module)}")
print(f"Parameters still accessible: {sum(p.numel() for p in model_ddp.parameters())}")

## 3. DistributedSampler -- Why shuffle=False and Sampler Handles It

`DistributedSampler` ensures each rank sees a disjoint shard of the dataset. It takes responsibility for shuffling, so the DataLoader itself must NOT shuffle (`shuffle=False`). Additionally, the sampler must be told the current epoch via `sampler.set_epoch(epoch)` before each epoch to change the random permutation -- without this, all epochs use the same data order.

In [ ]:
# DistributedSampler pattern
class NumberedDataset(Dataset):
    def __init__(self, n=100):
        self.data = torch.arange(n, dtype=torch.float32)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

ds = NumberedDataset(100)

# Single-rank simulation (rank=0, world_size=2, so we see half the data)
sampler = DistributedSampler(ds, num_replicas=2, rank=0, shuffle=True, seed=42)
loader_dist = DataLoader(
    ds, batch_size=10,
    sampler=sampler,  # sampler handles ordering
    shuffle=False,    # MUST be False when using sampler
)

print("Epoch 0 data (rank 0, world_size 2):")
sampler.set_epoch(0)  # CRITICAL: changes permutation each epoch
epoch0_data = [item.tolist() for item in loader_dist]
print(f"  Batches: {epoch0_data}")
print(f"  Total samples seen: {sum(len(b) for b in epoch0_data)} (= dataset_size / world_size = 50)")

print("\nEpoch 1 data (different order due to set_epoch(1)):")
sampler.set_epoch(1)
epoch1_data = [item.tolist() for item in loader_dist]
print(f"  First batch: {epoch1_data[0]}")
print(f"  Same as epoch 0?: {epoch0_data[0] == epoch1_data[0]}")

## 4. Gradient All-Reduce -- Ring Topology and Backward Overlap

Not all GPUs communicate directly with all other GPUs -- the NCCL all-reduce uses a **ring topology**: each GPU sends gradients to the next GPU in the ring, receives from the previous, and the information propagates in O(N) steps with O(1) per-step overhead per GPU regardless of world size.

The critical efficiency mechanism: DDP does NOT wait until all gradients are computed before starting communication. Instead, gradients are organised into **buckets** (default 25 MB each). As soon as a bucket is full (all its gradients are ready), DDP fires the all-reduce for that bucket while backward continues computing gradients for earlier layers. This overlaps computation and communication.

In [ ]:
# Demonstrate DDP bucket-based overlap with a logging hook
model_bucket = SimpleModel().to(device)
model_bucket_ddp = DDP(model_bucket, bucket_cap_mb=1)  # small buckets for demo

# Hook to observe when each gradient is reduced
reduction_order = []
for name, param in model_bucket_ddp.named_parameters():
    def make_hook(n):
        def hook(grad):
            reduction_order.append(n)
            return grad
        return hook
    param.register_hook(make_hook(name))

# Forward + backward
x_bkt = torch.randn(8, 16, device=device)
y_bkt = torch.randint(0, 4, (8,), device=device)
out = model_bucket_ddp(x_bkt)
loss = F.cross_entropy(out, y_bkt)
loss.backward()

print("Gradient reduction order (last layer first -- DDP starts reducing immediately):")
for i, name in enumerate(reduction_order):
    print(f"  {i+1}. {name}")
print("\nNote: gradients are reduced in reverse order (fc3 before fc1)")
print("This is why DDP achieves backward-communication overlap")

## 5. torchrun -- Launching Multi-Process Training

`torchrun` (replaces the older `torch.distributed.launch`) is the recommended way to launch DDP training. It:
- Spawns `nproc_per_node` processes per node
- Sets environment variables: `LOCAL_RANK`, `RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT`
- Handles fault tolerance and elastic scaling (with `rdzv_backend=c10d`)

The typical production script is a single Python file that runs correctly both single-GPU and multi-GPU, governed entirely by environment variables.

In [ ]:
# The canonical production DDP script structure (not executed, for reference)
ddp_script = '''
# train_ddp.py -- launch with:
# torchrun --nproc_per_node=8 train_ddp.py

import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DistributedSampler, DataLoader

def main():
    # torchrun sets these environment variables
    local_rank  = int(os.environ["LOCAL_RANK"])   # GPU index on this node
    rank        = int(os.environ["RANK"])          # global rank (0 to world_size-1)
    world_size  = int(os.environ["WORLD_SIZE"])    # total number of processes

    # Initialise process group (NCCL for GPU training)
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)

    # Build model, move to GPU BEFORE DDP
    model = MyModel().to(local_rank)
    model = DDP(model, device_ids=[local_rank])

    # Dataset with DistributedSampler
    dataset = MyDataset(...)
    sampler = DistributedSampler(dataset, rank=rank, num_replicas=world_size)
    loader  = DataLoader(dataset, batch_size=32, sampler=sampler, shuffle=False)

    for epoch in range(NUM_EPOCHS):
        sampler.set_epoch(epoch)  # CRITICAL: new shuffle order each epoch

        model.train()
        for X, y in loader:
            X, y = X.to(local_rank), y.to(local_rank)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X), y)
            loss.backward()  # triggers all-reduce automatically
            optimizer.step()

        # Save checkpoint only on rank 0
        if rank == 0:
            torch.save(model.module.state_dict(), f"ckpt_epoch{epoch}.pt")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''
print(ddp_script)

## 6. When DDP Breaks -- Common Failure Modes

DDP has sharp edges that cause training to diverge or crash silently.

In [ ]:
# Demonstrate common DDP pitfalls (all caught safely)

# Pitfall 1: BatchNorm with DDP -- BN uses per-rank batch stats, not global
# Solution: SyncBatchNorm synchronises BN stats across ranks
model_bn = nn.Sequential(nn.Linear(8, 16), nn.BatchNorm1d(16), nn.Linear(16, 4))
model_bn = nn.SyncBatchNorm.convert_sync_batchnorm(model_bn)
print("SyncBatchNorm conversion:")
for name, m in model_bn.named_modules():
    if isinstance(m, (nn.BatchNorm1d, nn.SyncBatchNorm)):
        print(f"  {name}: {type(m).__name__}")

# Pitfall 2: Accessing model.weight instead of model.module.weight inside DDP
model_w = SimpleModel().to(device)
ddp_w = DDP(model_w)
# Wrong: model.fc1 does not exist (DDP wraps the module)
try:
    _ = ddp_w.fc1
except AttributeError as e:
    print(f"\nPitfall 2: {e}")
    print("Fix: use ddp_w.module.fc1 to access the underlying module")
# Correct: use .module
print(f"  Correct: ddp_w.module.fc1 = {ddp_w.module.fc1}")

# Pitfall 3: Saving DDP model directly (saves with 'module.' prefix in keys)
sd = ddp_w.state_dict()
plain_keys = list(sd.keys())[:2]
print(f"\nDDP state_dict keys have 'module.' prefix: {plain_keys}")
print("Fix: save ddp_w.module.state_dict() to get standard keys")
clean_sd = ddp_w.module.state_dict()
print(f"module.state_dict() keys: {list(clean_sd.keys())[:2]}")

# Cleanup
if dist.is_initialized():
    dist.destroy_process_group()
    print("\nProcess group destroyed.")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| DDP paradigm | Replicate model on each rank; shard data; all-reduce gradients |
| init_process_group | Establishes communication group; nccl for GPU, gloo for CPU |
| DDP wrapping | Broadcasts rank-0 params; registers backward hooks for all-reduce |
| DistributedSampler | Shards dataset across ranks; set shuffle=False on DataLoader |
| set_epoch(epoch) | Required to change shuffle order each epoch |
| Bucket all-reduce | DDP reduces gradient buckets during backward; overlaps compute/communication |
| torchrun | Sets LOCAL_RANK, RANK, WORLD_SIZE; preferred launcher over distributed.launch |

### Common Pitfalls
- Not calling `sampler.set_epoch(epoch)` -- same data order every epoch, poor generalisation
- Saving `ddp_model.state_dict()` -- keys have 'module.' prefix; save `ddp_model.module.state_dict()`
- Using `nn.BatchNorm` without `SyncBatchNorm` -- each rank normalises its own shard independently
- Moving model to device AFTER DDP wrapping -- DDP cannot find the right device
- Using `shuffle=True` on DataLoader with DistributedSampler -- overrides sampler, breaks sharding

### Next: Notebook 12 -- Capstone Production Pipeline